## Partie A : Les données

In [57]:
import numpy as np

#### Exemple d'observations

In [58]:
observations = [
    [("B", "T", "G"), ("B", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("B", "T", "P")],
    [("D", "T", "G"), ("D", "T", "G"), ("D", "T", "P")],
    [("B", "T", "G"), ("B", "T", "G"), ("D", "T", "P")],
    [("B", "S", "G"), ("B", "S", "G"), ("D", "S", "P")],
    [("B", "S", "G"), ("B", "S", "G"), ("B", "S", "P")],
    [("D", "S", "G"), ("D", "S", "G"), ("D", "S", "P")]
]
print(observations)

[[('B', 'T', 'G'), ('B', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('B', 'T', 'P')], [('D', 'T', 'G'), ('D', 'T', 'G'), ('D', 'T', 'P')], [('B', 'T', 'G'), ('B', 'T', 'G'), ('D', 'T', 'P')], [('B', 'S', 'G'), ('B', 'S', 'G'), ('D', 'S', 'P')], [('B', 'S', 'G'), ('B', 'S', 'G'), ('B', 'S', 'P')], [('D', 'S', 'G'), ('D', 'S', 'G'), ('D', 'S', 'P')]]


#### Matrice aléatoire (n rangées de triplets et 3 colonnes)

In [59]:
def generate_alea(n):
    """
    - C = Cheveux ∈ {B, D}  (Blond, Dark)
    - H = Hauteur ∈ {T, S}  (Tall, Short)
    - P = Pays ∈ {G, P}     (Greenland, Poland)
    """
    tableau = []
    for i in range(n):
        ligne = []
        for j in range(3):
            x1 = np.random.choice(['B', 'D'])
            x2 = np.random.choice(['T', 'S'])
            x3 = np.random.choice(['G', 'P'])
            ligne.append((x1,x2,x3))
        tableau.append(ligne)
    return tableau
tableau_alea = generate_alea(6)
print(tableau_alea)

[[('D', 'T', 'P'), ('B', 'T', 'P'), ('D', 'S', 'G')], [('D', 'T', 'G'), ('B', 'S', 'G'), ('D', 'S', 'P')], [('B', 'T', 'G'), ('D', 'T', 'G'), ('B', 'T', 'P')], [('B', 'T', 'G'), ('B', 'T', 'P'), ('D', 'S', 'G')], [('D', 'S', 'G'), ('D', 'T', 'P'), ('D', 'S', 'P')], [('D', 'T', 'G'), ('B', 'S', 'G'), ('B', 'T', 'G')]]


## Partie B : Question

*"Si vous observez un nouvel individu grand ayant cheveux blonds, quel est son pays d’origine le plus probable?"*

Il s'agit de comparer P(Greenland|B,T) avec P(Poland|B,T)

#### Matrice N lignes dont chaque élément est constitué d’un triplet : (X1, X2, X3), Xi ≠ Xj, Xi ∈ {D, T, G}

In [60]:
def generate_DTG(n):
    table = []
    choix = ['D','T','G']
    for _ in range(n):
        triplet = np.random.choice(choix, size=3, replace=False)
        triplet_liste = list(triplet)
        table.append(triplet_liste)
    return table
print(generate_DTG(10))

[['D', 'T', 'G'], ['G', 'T', 'D'], ['D', 'G', 'T'], ['D', 'T', 'G'], ['G', 'D', 'T'], ['D', 'G', 'T'], ['G', 'D', 'T'], ['T', 'D', 'G'], ['G', 'D', 'T'], ['G', 'D', 'T']]


## Partie C : Maximum a posteriori

### 1. MAP dans le cadre Naïf de Bayes

In [ ]:
def map_naif_bayes(data):
    
    total_g = 0; compte_b_g = 0; compte_t_g = 0
    total_p = 0; compte_b_p = 0; compte_t_p = 0
    total = 0
    
    for row in data:
        for triplet in row:
            total += 1
            b_obs, t_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            if pays_obs == 'G':
                total_g += 1
                if b_obs == 'B': compte_b_g += 1
                if t_obs == 'T': compte_t_g += 1
            elif pays_obs == 'P':
                total_p += 1
                if b_obs == 'B': compte_b_p += 1
                if t_obs == 'T': compte_t_p += 1
                
    p_g = total_g / total if total > 0 else 0
    p_p = total_p / total if total > 0 else 0
    
    p_b_g = compte_b_g / total_g if total_g > 0 else 0
    p_t_g = compte_t_g / total_g if total_g > 0 else 0
    p_b_p = compte_b_p / total_p if total_p > 0 else 0
    p_t_p = compte_t_p / total_p if total_p > 0 else 0
    
    vraisemblance_g = p_b_g * p_t_g
    vraisemblance_p = p_b_p * p_t_p
    
    score_g = vraisemblance_g * p_g
    score_p = vraisemblance_p * p_p
    
    return {
        'P(B|G)': p_b_g,
        'P(T|G)': p_t_g,
        'P(B,T|G)': vraisemblance_g,
        'P(G)': p_g,
        'Score(G)': score_g,
        'P(B|P)': p_b_p,
        'P(T|P)': p_t_p,
        'P(B,T|P)': vraisemblance_p,
        'P(P)': p_p,
        'Score(P)': score_p,
        'Prediction': 'Greenland' if score_g > score_p else 'Poland'
    }

#### Affichage des résultats

In [ ]:
print("Resultat MAP (Naïf)")
resultat = map_naif_bayes(tableau_alea)
for cle, valeur in resultat.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

Resultat MAP (Naïf)
P(B|G) : 0.4545
P(T|G) : 0.5455
P(B,T|G) : 0.2479
P(G) : 0.6111
Score(G) : 0.1515
P(B|P) : 0.4286
P(T|P) : 0.7143
P(B,T|P) : 0.3061
P(P) : 0.3889
Score(P) : 0.1190
Prediction : Greenland


### 2. MAP sans le cadre Naïf de Bayes

In [ ]:
def map_no_naif_bayes(data):
    
    total_g = 0; compte_b_t_g = 0
    total_p = 0; compte_b_t_p = 0
    total = 0
    
    for row in data:
        for triplet in row:
            total += 1
            b_obs, t_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if b_obs == 'B' and t_obs == 'T': compte_b_t_g += 1
            elif pays_obs == 'P':
                total_p += 1
                if b_obs == 'B' and t_obs == 'T': compte_b_t_p += 1

    p_g = total_g / total if total > 0 else 0
    p_p = total_p / total if total > 0 else 0
    
    vraisemblance_g = compte_b_t_g / total_g if total_g > 0 else 0
    vraisemblance_p = compte_b_t_p / total_p if total_p > 0 else 0
    
    score_g = vraisemblance_g * p_g
    score_p = vraisemblance_p * p_p
    
    return {
            'P(B,T|G)': vraisemblance_g,
            'P(G)': p_g,
            'Score(G)': score_g,
            'P(B,T|P)': vraisemblance_p,
            'P(P)': p_p,
            'Score(P)': score_p,
            'Count(B,T in G)': compte_b_t_g,
            'Count(B,T in P)': compte_b_t_p,
            'Prediction': 'Greenland' if score_g > score_p else 'Poland'
        }

#### Affichage des résultats

In [ ]:
print("Resultat MAP (Sans Naïf)")
resultat_sans_naif = map_no_naif_bayes(tableau_alea)
for cle, valeur in resultat_sans_naif.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

Resultat MAP (Sans Naïf)
P(B,T|G) : 0.2727
P(G) : 0.6111
Score(G) : 0.1667
P(B,T|P) : 0.4286
P(P) : 0.3889
Score(P) : 0.1667
Count(B,T in G) : 3
Count(B,T in P) : 3
Prediction : Poland


## Partie D : Maximum de vraisemblance

MLE Sans Naïf (Conjoint direct) : $$Vraisemblance = P(Cheveux, Hauteur \vert{} Pays)$$

MLE Naïf (Indépendance) : $$Vraisemblance = P(Cheveux \vert{} Pays) \times P(Hauteur \vert{} Pays)$$

#### MLE avec le cadre Naïf Bayes

In [80]:
def mle_naif_bayes(data):

    total_g = 0; compte_b_g = 0; compte_t_g = 0
    total_p = 0; compte_b_p = 0; compte_t_p = 0
    
    for row in data:
        for triplet in row:
            b_obs, t_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if b_obs == 'B': compte_b_g += 1
                if t_obs == 'T': compte_t_g += 1
                    
            elif pays_obs == 'P':
                total_p += 1
                if b_obs == 'B': compte_b_p += 1
                if t_obs == 'T': compte_t_p += 1
                
    p_b_g = compte_b_g / total_g if total_g > 0 else 0
    p_t_g = compte_t_g / total_g if total_g > 0 else 0
    p_b_p = compte_b_p / total_p if total_p > 0 else 0
    p_t_p = compte_t_p / total_p if total_p > 0 else 0
    
    likelihood_g = p_b_g * p_t_g
    likelihood_p = p_b_p * p_t_p
    
    return {
        'P(B|G)': p_b_g,
        'P(T|G)': p_t_g,
        'P(B,T|G)': likelihood_g,
        'P(B|P)': p_b_p,
        'P(T|P)': p_t_p,
        'P(B,T|P)': likelihood_p,
        'Prediction': 'Greenland' if likelihood_g > likelihood_p else 'Poland'
    }

#### Affichage des résultats

In [84]:
print("Resultat MLE (Naïf)\n")
resultat_naif = mle_naif_bayes(tableau_alea)
for cle, valeur in resultat_naif.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

Resultat MLE (Naïf)

P(B|G) : 0.4545
P(T|G) : 0.5455
P(B,T|G) : 0.2479
P(B|P) : 0.4286
P(T|P) : 0.7143
P(B,T|P) : 0.3061
Prediction : Poland


#### MLE sans le cadre Naîf Bayes

In [73]:
def mle_no_naif_bayes(data):
    
    total_g = 0
    total_p = 0
    compte_b_t_g = 0
    compte_b_t_p = 0
    
    for row in data:
        for triplet in row:
            b_obs, t_obs, pays_obs = triplet[0], triplet[1], triplet[2]
            
            if pays_obs == 'G':
                total_g += 1
                if b_obs == 'B' and t_obs == 'T':
                    compte_b_t_g += 1
                    
            elif pays_obs == 'P':
                total_p += 1
                if b_obs == 'B' and t_obs == 'T':
                    compte_b_t_p += 1
                    
    likelihood_g = compte_b_t_g / total_g if total_g > 0 else 0
    likelihood_p = compte_b_t_p / total_p if total_p > 0 else 0
    
    return {
        'P(B,T|G)': likelihood_g,
        'P(B,T|P)': likelihood_p,
        'Count(B,T in G)': compte_b_t_g,
        'Count(B,T in P)': compte_b_t_p,
        'Prediction': 'Greenland' if likelihood_g > likelihood_p else 'Poland'
    }


#### Affichage des résultats

In [85]:
print("Resultat MLE (Non Naïf)\n")
resultat_non_naif = mle_naif_bayes(tableau_alea)
for cle, valeur in resultat_non_naif.items():
    if isinstance(valeur, float):
        print(f"{cle} : {valeur:.4f}")
    else:
        print(f"{cle} : {valeur}")

Resultat MLE (Non Naïf)

P(B|G) : 0.4545
P(T|G) : 0.5455
P(B,T|G) : 0.2479
P(B|P) : 0.4286
P(T|P) : 0.7143
P(B,T|P) : 0.3061
Prediction : Poland


## Partie E :

Question 1 :

Pour passer à des valeurs continues on doit remplacer np.random.choice(['B', 'D']) par np.random.uniform(0, 1) pour tirer un nombre flottant.

Question 2 :

On pourrait découper (0,1) en deux intervalles :
- Au dessus de 0,5 : dark
- En dessous de 0,5 : blond

Question 3 :

Au lieu de vérifier si la variable correspond à la lettre, on change les conditions pour tester la valeur en faisant if c_obs < 0.5 (et if c_obs >= 0.5 pour dark)